In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/kerala_rainfall_1901_2017.csv")

print("Dataset shape:", df.shape)
print("Year range:", df["YEAR"].min(), "-", df["YEAR"].max())
print("Unique years:", df["YEAR"].nunique())

Dataset shape: (117, 19)
Year range: 1901 - 2017
Unique years: 117


In [2]:
flood_years = [1924, 1958, 1961, 1992, 2007, 2013]

print("Selected historical flood years:", flood_years)

Selected historical flood years: [1924, 1958, 1961, 1992, 2007, 2013]


In [3]:
df["FLOOD"] = df["YEAR"].isin(flood_years).astype(int)

df[["YEAR", "FLOOD"]].head()

,YEAR,FLOOD
0,1901,0
1,1902,0
2,1903,0
3,1904,0
4,1905,0


In [4]:
df[df["FLOOD"] == 1][["YEAR", "FLOOD"]]

,YEAR,FLOOD
23,1924,1
57,1958,1
60,1961,1
91,1992,1
106,2007,1
112,2013,1


In [5]:
print(df["FLOOD"].value_counts())

FLOOD
0    111
1      6
Name: count, dtype: int64


In [6]:
print(df["FLOOD"].value_counts(normalize=True) * 100)

FLOOD
0    94.871795
1     5.128205
Name: proportion, dtype: float64


In [7]:
df.to_csv(
    "../data/processed/kerala_flood_labelled.csv",
    index=False
)

print("Labelled dataset saved successfully!")

Labelled dataset saved successfully!


In [9]:
processed_df = pd.read_csv(
    "../data/processed/kerala_flood_labelled.csv"
)

print("Shape:", processed_df.shape)
print("\nColumns:", processed_df.columns.tolist())
print("\nFlood distribution:")
print(processed_df["FLOOD"].value_counts())

Shape: (117, 20)

Columns: ['SUBDIVISION', 'YEAR', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'ANNUAL', 'JF', 'MAM', 'JJAS', 'OND', 'FLOOD']

Flood distribution:
FLOOD
0    111
1      6
Name: count, dtype: int64


In [10]:
months = [
    "JAN", "FEB", "MAR", "APR", "MAY",
    "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"
]

feature_windows = {
    "January-March": months[:3],
    "January-April": months[:4],
    "January-May": months[:5]
}

for name, features in feature_windows.items():
    print(f"\n{name}")
    print("Features:", features)
    print("Number of features:", len(features))


January-March
Features: ['JAN', 'FEB', 'MAR']
Number of features: 3

January-April
Features: ['JAN', 'FEB', 'MAR', 'APR']
Number of features: 4

January-May
Features: ['JAN', 'FEB', 'MAR', 'APR', 'MAY']
Number of features: 5


In [11]:
# Select January-March rainfall as input features
X_jan_mar = df[feature_windows["January-March"]].copy()

# Select the target variable
y = df["FLOOD"].copy()

print("Feature matrix shape:", X_jan_mar.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X_jan_mar.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

Feature matrix shape: (117, 3)
Target shape: (117,)

Feature columns:
['JAN', 'FEB', 'MAR']

Target distribution:
FLOOD
0    111
1      6
Name: count, dtype: int64


In [12]:
# January-April feature matrix
X_jan_apr = df[feature_windows["January-April"]].copy()

# January-May feature matrix
X_jan_may = df[feature_windows["January-May"]].copy()

# Display shapes
print("January-March:", X_jan_mar.shape)
print("January-April:", X_jan_apr.shape)
print("January-May:", X_jan_may.shape)

January-March: (117, 3)
January-April: (117, 4)
January-May: (117, 5)


In [13]:
# Sort observations by year
df = df.sort_values("YEAR").reset_index(drop=True)

# Display all documented flood years
print("Documented flood years:")
print(df.loc[df["FLOOD"] == 1, ["YEAR", "FLOOD"]])

# Examine a possible 70:30 chronological split
split_index = int(len(df) * 0.70)

train_preview = df.iloc[:split_index]
test_preview = df.iloc[split_index:]

print("\nPotential training period:")
print(train_preview["YEAR"].min(), "-", train_preview["YEAR"].max())
print("Training flood years:", train_preview["FLOOD"].sum())

print("\nPotential testing period:")
print(test_preview["YEAR"].min(), "-", test_preview["YEAR"].max())
print("Testing flood years:", test_preview["FLOOD"].sum())

Documented flood years:
     YEAR  FLOOD
23   1924      1
57   1958      1
60   1961      1
91   1992      1
106  2007      1
112  2013      1

Potential training period:
1901 - 1981
Training flood years: 3

Potential testing period:
1982 - 2017
Testing flood years: 3


In [14]:
split_index = round(len(df) * 0.70)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Training dataset:")
print("Shape:", train_df.shape)
print("Period:", train_df["YEAR"].min(), "-", train_df["YEAR"].max())
print("Flood years:", train_df["FLOOD"].sum())

print("\nTesting dataset:")
print("Shape:", test_df.shape)
print("Period:", test_df["YEAR"].min(), "-", test_df["YEAR"].max())
print("Flood years:", test_df["FLOOD"].sum())

Training dataset:
Shape: (82, 20)
Period: 1901 - 1982
Flood years: 3

Testing dataset:
Shape: (35, 20)
Period: 1983 - 2017
Flood years: 3


In [15]:
from sklearn.model_selection import StratifiedKFold

# Use January-May features for this initial check
X_train = train_df[feature_windows["January-May"]]
y_train = train_df["FLOOD"]

# Create 3-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train), start=1
):

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    print(f"\nFold {fold}")
    print("Training observations:", len(train_idx))
    print("Training flood years:", y_fold_train.sum())
    print("Validation observations:", len(val_idx))
    print("Validation flood years:", y_fold_val.sum())


Fold 1
Training observations: 54
Training flood years: 2
Validation observations: 28
Validation flood years: 1

Fold 2
Training observations: 55
Training flood years: 2
Validation observations: 27
Validation flood years: 1

Fold 3
Training observations: 55
Training flood years: 2
Validation observations: 27
Validation flood years: 1
